IMPORTS

In [1]:
import os, time, json, copy, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

import torch
import torch.nn as nn
import torch.optim as optim
from torch.optim.lr_scheduler import CosineAnnealingLR
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader, random_split
# Slúži na potláčanie zbytočných chýb
warnings.filterwarnings("ignore")
# Test či načítalo CUDU
print(f"PyTorch: {torch.__version__}")
print(f"CUDA dostupná: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")


PyTorch: 2.6.0+cu124
CUDA dostupná: True
GPU: NVIDIA GeForce GTX 1650
VRAM: 4.3 GB


KONFIGURÁCIA

In [ ]:
# Cesty
DATA_DIR    = "./data"
RESULTS_DIR = "./results"

# Hyperparametre
INPUT_SIZE   = 224     # VGG16 vstup
BATCH_SIZE   = 64
NUM_EPOCHS   = 30
NUM_CLASSES  = 10
VAL_SPLIT    = 0.1     # 10 % z train = validácia
SEED         = 42

# Learning rates
LR_HEAD      = 1e-3    # klasifikácia
LR_BACKBONE  = 1e-4    # konvolučné vrstvy
WEIGHT_DECAY = 1e-4

# Triedy CIFAR-10
CIFAR10_CLASSES = ["airplane", "automobile", "bird", "cat", "deer",
                   "dog", "frog", "horse", "ship", "truck"]

torch.manual_seed(SEED)
np.random.seed(SEED)


DATASET

In [ ]:
# ImageNet normalizácia - RGB normalizujeme
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

# Transformácie
train_transform = transforms.Compose([
    transforms.Resize((INPUT_SIZE, INPUT_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

val_test_transform = transforms.Compose([
    transforms.Resize((INPUT_SIZE, INPUT_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

no_aug_transform = transforms.Compose([
    transforms.Resize((INPUT_SIZE, INPUT_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

